In [8]:
import nest_asyncio
nest_asyncio.apply()

In [9]:
import asyncio
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import httpx
from bs4 import BeautifulSoup
from contextlib import asynccontextmanager
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE       = "https://www.arrow.com"
CAT_URL    = f"{BASE}/en/categories"
SEARCH_API = f"{BASE}/productsearch/productlinesearchresultajax"
PER_PAGE   = 50
OUT_FILE   = Path("discovery.json")

HEADERS = {
    "user-agent":                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "accept":                    "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "accept-language":           "en-US,en;q=0.9",
    "accept-encoding":           "gzip, deflate, br",
    "cache-control":             "no-cache",
    "pragma":                    "no-cache",
    "sec-ch-ua":                 '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "sec-ch-ua-mobile":          "?0",
    "sec-ch-ua-platform":        '"Windows"',
    "sec-fetch-dest":            "document",
    "sec-fetch-mode":            "navigate",
    "sec-fetch-site":            "none",
    "sec-fetch-user":            "?1",
    "upgrade-insecure-requests": "1",
}

In [10]:
BD_HOST = "brd.superproxy.io"
BD_PORT = 33335
BD_USER = "brd-customer-hl_14d32bce-zone-residential_proxy1"#"brd-customer-hl_14d32bce-zone-datacenter_proxy1"
BD_PASS = "d0fh393blkuv"#"ymg5cg3a1z33"#


def get_proxy_url() -> str:
    return f"http://{BD_USER}:{BD_PASS}@{BD_HOST}:{BD_PORT}"

In [11]:
def slugify(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")

def build_pdp_url(part_number: str, manufacturer: str) -> str:
    return f"{BASE}/en/products/{quote(part_number)}/{slugify(manufacturer)}"

def load_existing() -> dict:
    if OUT_FILE.exists():
        data = json.loads(OUT_FILE.read_text())
        print(f"[load] resuming — {len(data['categories'])} categories already in file")
        return data
    print("[load] no existing file, starting fresh")
    return {"meta": {}, "categories": {}}

def save(data: dict):
    total_cats  = len(data["categories"])
    total_prods = sum(v["total_products"] for v in data["categories"].values())
    OUT_FILE.write_text(json.dumps(data, indent=2))
    print(f"[save] wrote {OUT_FILE} — {total_cats} categories, {total_prods} products")

In [12]:
@asynccontextmanager
async def fresh_client(cookies: dict = None):
    async with httpx.AsyncClient(
        proxy=get_proxy_url(),
        headers=HEADERS,
        cookies=cookies or {},
        timeout=30,
        follow_redirects=True,
    ) as client:
        yield client

In [13]:
async def get_categories_html_with_client(client: httpx.AsyncClient) -> str:
    """
    Load /en/categories using provided httpx client.
    Returns HTML for category parsing.
    Client automatically maintains cookies.
    """
    print(f"[httpx] fetching → {CAT_URL}")
    try:
        r = await client.get(CAT_URL)
        print(f"[httpx] HTTP {r.status_code} — {len(r.text)} chars")
        print(f"[httpx] client now has {len(client.cookies.jar)} cookies")
        return r.text
    except Exception as e:
        print(f"[httpx] ERROR: {type(e).__name__}: {e}")
        raise

In [14]:
def parse_categories(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "html.parser")
    cats = []
    seen = set()

    for h2 in soup.select("h2.Sitemap-heading"):
        h2_a       = h2.select_one("a")
        h2_name    = h2_a.get_text(strip=True) if h2_a else ""
        parent_div = h2.find_parent("div")

        # flat leaf items directly under h2 (no subheading)
        for ul in parent_div.select("ul.Sitemap-List"):
            if ul.find_parent("li", class_="Sitemap-List-item") and ul.find_parent("h3"):
                continue
            for li in ul.select("li.Sitemap-List-item"):
                if li.select_one("h3.Sitemap-subHeading"):
                    continue
                a    = li.select_one("a")
                href = a.get("href", "") if a else ""
                name = a.get_text(strip=True) if a else ""
                if not href or href in seen:
                    continue
                seen.add(href)
                parts = href.strip("/").split("/")
                if len(parts) < 4:
                    print(f"  [skip] {href}")
                    continue
                cats.append({
                    "name":        name,
                    "slug":        parts[-1],
                    "parent":      parts[-2],
                    "url":         BASE + href,
                    "prodline":    name,
                    "h2_category": h2_name,
                    "h3_category": None,
                })
                print(f"  [flat] {parts[-2]}/{parts[-1]} → '{name}'")

        # nested items under h3 subheadings
        for h3 in parent_div.select("h3.Sitemap-subHeading"):
            h3_a      = h3.select_one("a")
            h3_name   = h3_a.get_text(strip=True) if h3_a else ""
            nested_ul = (
                h3.find_next_sibling("ul")
                or (h3.find_parent("li") and h3.find_parent("li").select_one("ul.u-indent-left"))
            )
            if not nested_ul:
                continue
            for li in nested_ul.select("li.Sitemap-List-item"):
                a    = li.select_one("a")
                href = a.get("href", "") if a else ""
                name = a.get_text(strip=True) if a else ""
                if not href or href in seen:
                    continue
                seen.add(href)
                parts = href.strip("/").split("/")
                if len(parts) < 4:
                    print(f"  [skip] {href}")
                    continue
                cats.append({
                    "name":        name,
                    "slug":        parts[-1],
                    "parent":      parts[-2],
                    "url":         BASE + href,
                    "prodline":    name,
                    "h2_category": h2_name,
                    "h3_category": h3_name,
                })
                print(f"  [sub]  {parts[-2]}/{parts[-1]} → '{name}' (under {h2_name} > {h3_name})")

    print(f"\n[categories] parsed {len(cats)} leaf categories")
    return cats

In [46]:
def parse_products_from_api_response(data: dict) -> list[dict]:
    products = []
    items = data.get("products", [])

    for item in items:
        try:
            pn      = item.get("fullPart") or item.get("fullPartOriginal")
            mfr     = item.get("mfrName")
            part_id = item.get("partId")
            seo     = item.get("partSeo")
            mfr_seo = item.get("mfrSeo")

            if not pn or not mfr:
                continue

            # ── pricing ──────────────────────────────────────────
            gbp     = item.get("groupByPrice") or {}
            regions = gbp.get("regionalBuyingOptions") or []

            lowest_price = None
            total_stock  = gbp.get("totalStock", 0)

            if regions:
                # take NAC first, fall back to first available
                nac = next((r for r in regions if r.get("inventoryRegion") == "NAC"), regions[0])
                lowest_price = (nac.get("lowestPrice") or {}).get("price")

            # ── features as flat dict ─────────────────────────────
            feat = {
                f["name"]: (f["value"] + (" " + f["uom"] if f.get("uom") else "")).strip()
                for f in item.get("features", [])
                if f.get("value")
            }

            pdp_url = (
                f"https://www.arrow.com/en/products/{mfr_seo}/{seo}"
                if mfr_seo and seo else None
            )
            datasheet_url = (item.get("datasheet") or {}).get("url")
            if datasheet_url and datasheet_url.startswith("//"):
                datasheet_url = "https:" + datasheet_url
            image_url = item.get("smallImageUrl", "")
            if image_url and image_url.startswith("//"):
                image_url = "https:" + image_url

            products.append({
                "partNumber":    pn,
                "manufacturer":  mfr,
                "arrowId":       part_id,
                "description":   item.get("storedPartDescription"),
                "productLine":   item.get("productLineName"),
                "partStatus":    item.get("partStatus"),
                "eccnCode":      item.get("eccnCode", "EAR99"),
                "euRohs":        item.get("euRohs"),
                "military":      item.get("militaryFlag") == "true",
                "lowestPrice":   lowest_price,
                "totalStock":    total_stock,
                "datasheetUrl":  datasheet_url,
                "imageUrl":      image_url,
                "pdpUrl":        pdp_url,
                "features":      feat,
            })

        except Exception as e:
            print(f"  [warn] skipped {item.get('fullPart','?')}: {e}")
            continue

    return products

--- 

### PLP


In [49]:
async def bootstrap_arrow(category_slug: str, use_proxy: bool = True):
    global PW, BROWSER, CONTEXT, PAGE, CAPTURED_PRODUCTS

    CAPTURED_PRODUCTS = []
    category_url = f"https://www.arrow.com/en/search-result.html?cat={category_slug}"

    print("1. Launching Playwright Chromium...")
    PW = await async_playwright().start()

    launch_kwargs = dict(
        headless=False,
        args=[
            "--disable-blink-features=AutomationControlled",
            "--ignore-certificate-errors",
        ],
    )
    if use_proxy:
        launch_kwargs["proxy"] = {
            "server":   f"http://{BD_HOST}:{BD_PORT}",
            "username": BD_USER,
            "password": BD_PASS,
        }

    BROWSER = await PW.chromium.launch(**launch_kwargs)

    CONTEXT = await BROWSER.new_context(
        ignore_https_errors=True,
        user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
        viewport={"width": 1280, "height": 800},
    )

    PAGE = await CONTEXT.new_page()

    async def on_response(resp):
        if "experienceservices/search" in resp.url and resp.request.method == "POST":
            try:
                body = await resp.body()
                if not body:
                    return
                data = json.loads(body)
                n_prods = len(data.get("products", []))
                if n_prods == 0:
                    return
                CAPTURED_PRODUCTS.append(data)
                pagination = data.get("pagination", {})
                print(
                    f"  [intercept] ✅ {resp.status} — "
                    f"{n_prods} products | "
                    f"page {pagination.get('currentPage')}/{pagination.get('totalPages')} | "
                    f"total {pagination.get('totalResults')}"
                )
            except Exception as e:
                print(f"  [intercept] error: {e}")

    PAGE.on("response", on_response)

    print(f"2. Navigating to {category_url} {'(proxy)' if use_proxy else '(direct)'}")
    try:
        await PAGE.goto(category_url, wait_until="domcontentloaded", timeout=30_000)
        print("3. DOM loaded")
    except Exception as e:
        print(f"3. Navigation exception (continuing): {e}")

    print("4. Waiting 15s for Akamai JS + product API to fire...")
    await asyncio.sleep(15)

    title = await PAGE.title()
    print(f"5. Page title: {title}")

    cookies_raw = await CONTEXT.cookies()
    print(f"6. {len(cookies_raw)} cookies collected")
    print(f"7. API responses captured: {len(CAPTURED_PRODUCTS)}")

    if CAPTURED_PRODUCTS:
        prods = parse_products_from_api_response(CAPTURED_PRODUCTS[-1])
        print(f"8. Products on page 1: {len(prods)}")
        if prods:
            print(f"   Sample: {prods[0]['partNumber']} | {prods[0]['manufacturer']}")
    else:
        print("8. ⚠️  No API response captured")

    print("\nBrowser staying open for pagination cell...")

# ── run — toggle use_proxy as needed ─────────────────────────────
await bootstrap_arrow("active-filters", use_proxy=True)

1. Launching Playwright Chromium...
2. Navigating to https://www.arrow.com/en/search-result.html?cat=active-filters (proxy)
3. DOM loaded
4. Waiting 15s for Akamai JS + product API to fire...
  [intercept] ✅ 200 — 25 products | page 1/23 | total 553
5. Page title: Search Result
6. 37 cookies collected
7. API responses captured: 1
8. Products on page 1: 25
   Sample: LTC1563-2IGN#PBF | Analog Devices

Browser staying open for pagination cell...


In [56]:
async def paginate_with_cookies(cat: dict, max_pages: int = None) -> list[dict]:
    global CONTEXT, CAPTURED_PRODUCTS

    all_products = []

    if not CAPTURED_PRODUCTS:
        print("[error] no page 1 data — run bootstrap first")
        return []

    # ── page 1 from browser intercept ────────────────────────────
    p1_data  = CAPTURED_PRODUCTS[-1]
    p1_prods = parse_products_from_api_response(p1_data)
    all_products.extend(p1_prods)

    pagination  = p1_data.get("pagination", {})
    total_count = pagination.get("totalResults", 0)
    total_pages = pagination.get("totalPages", 1)
    page_size   = pagination.get("pageSize", 25)

    if max_pages:
        total_pages = min(total_pages, max_pages)

    print(f"[paginate] {total_count} products / {total_pages} pages — page 1: {len(p1_prods)} products")

    if total_pages <= 1:
        print("[paginate] single page — done")
        return all_products

    headers = {
        "accept":         "application/json, text/plain, */*",
        "content-type":   "application/json",
        "origin":         "https://www.arrow.com",
        "referer":        f"https://www.arrow.com/en/search-result.html?cat={cat['slug']}",
        "sec-fetch-dest": "empty",
        "sec-fetch-mode": "cors",
        "sec-fetch-site": "same-origin",
        "dnt":            "1",
    }

    async with AsyncSession(impersonate="chrome120") as session:
        for page_num in range(2, total_pages + 1):

            # ── refresh cookies from live browser on every page ───
            raw_cookies = await CONTEXT.cookies()
            cookies     = {c["name"]: c["value"] for c in raw_cookies}

            payload = {
                "pageSize":      str(page_size),
                "currentPage":   str(page_num),
                "category":      cat["slug"],
                "manufacturer":  [],
                "searchTerm":    "",
                "sort":          "",
                "sortDirection": "",
                "filters":       [],
            }

            print(f"  [{page_num}/{total_pages}] POST page {page_num}...")

            try:
                r = await session.post(
                    SEARCH_API_NEW,
                    params=SEARCH_PARAMS,
                    json=payload,
                    headers=headers,
                    cookies=cookies,
                )
                print(f"  [{page_num}] HTTP {r.status_code}")

                if r.status_code != 200:
                    print(f"  [{page_num}] unexpected status — stopping")
                    break

                data       = r.json()
                raw_count  = len(data.get("products", []))
                page_prods = parse_products_from_api_response(data)
                all_products.extend(page_prods)
                print(f"  [{page_num}/{total_pages}] raw={raw_count} parsed={len(page_prods)} → {len(all_products)} total")

                if raw_count == 0:
                    print(f"  [{page_num}] empty response — stopping")
                    break

                await asyncio.sleep(0.7)

            except Exception as e:
                print(f"  [{page_num}] error: {type(e).__name__}: {e}")
                break

    print(f"[paginate] done — {len(all_products)}/{total_count} products collected")
    if len(all_products) < total_count:
        print(f"  ⚠️  missing {total_count - len(all_products)} products")

    return all_products

In [51]:
import json
from pathlib import Path

OUTPUT_FILE = Path("discovery.json")

def save_discovery(products: list[dict], output_file: Path = OUTPUT_FILE):
    """Append products to discovery.json, deduplicating by partNumber+manufacturer."""
    existing = []
    if output_file.exists():
        with open(output_file) as f:
            existing = json.load(f)

    # dedupe by (partNumber, manufacturer)
    seen = {(p["partNumber"], p["manufacturer"]) for p in existing}
    new  = [p for p in products if (p["partNumber"], p["manufacturer"]) not in seen]

    combined = existing + new
    with open(output_file, "w") as f:
        json.dump(combined, f, indent=2)

    print(f"[save] +{len(new)} new | {len(existing)} existing | {len(combined)} total → {output_file}")
    return combined

# ── run after paginate ────────────────────────────────────────────
save_discovery(products)

[save] +11 new | 50 existing | 61 total → discovery.json


[{'partNumber': 'LTC1569IS8-6#PBF',
  'manufacturer': 'Analog Devices',
  'arrowId': '41405737',
  'description': 'Active Filter Single Low Pass 10th Order 64kHz 8-Pin SOIC N Tube',
  'productLine': 'Active Filters',
  'partStatus': 'Active',
  'eccnCode': 'EAR99',
  'euRohs': 'Compliant',
  'military': False,
  'lowestPrice': 12.64,
  'totalStock': 100,
  'datasheetUrl': 'https://static6.arrow.com/aropdfconversion/8d366b0620c2f2c3a126185037eeb7e6f3b69507/15696f.pdf',
  'imageUrl': 'https://static6.arrow.com/aropdfconversion/arrowimages/e009de7b3c93724457c213360f6a4fd2650dbe89/soicn.jpg',
  'pdpUrl': 'https://www.arrow.com/en/products/analog-devices/ltc1569is8-6pbf',
  'features': {'Type': 'Low Pass Filter',
   'Maximum Order': '10th',
   'Number of Circuits': '1',
   'Cut-Off Frequency Range': '<100kHz',
   'Power Supply Type': 'Single|Dual',
   'Packaging': 'Tube',
   'Minimum Single Supply Voltage': '3 V',
   'Minimum Dual Supply Voltage': '±1.5 V',
   'Maximum Single Supply Voltage

In [52]:
async def close_browser():
    global PW, BROWSER, CONTEXT, PAGE
    if PAGE:    await PAGE.close()
    if CONTEXT: await CONTEXT.close()
    if BROWSER: await BROWSER.close()
    if PW:      await PW.stop()
    print("[closed] browser shut down")

await close_browser()

[closed] browser shut down


---

## Run Discovery, CATs only

In [ ]:
async def run_category_discovery():
    """
    One-time run: fetch /en/categories, parse all leaf categories,
    save to categories.json. No product scraping.
    """
    CAT_OUT = Path("categories.json")

    async with fresh_client() as client:
        html = await get_categories_html_with_client(client)

    cats = parse_categories(html)

    payload = {
        "meta": {
            "total_categories": len(cats),
            "scraped_at":       datetime.now(timezone.utc).isoformat(),
        },
        "categories": cats,
    }
    CAT_OUT.write_text(json.dumps(payload, indent=2))
    print(f"\n[saved] {CAT_OUT} — {len(cats)} categories")
    return cats

cats = await run_category_discovery()

--- 

### Test run, 2 cats, 2 pages each

In [54]:
async def run_test(n_cats: int = 2, n_pages: int = 2, use_proxy: bool = False):
    """
    Load categories from categories.json, scrape the first n_cats
    with a max_pages cap using the new experienceservices API.
    Saves results to 2cat2page_test.json.
    """
    CAT_FILE    = Path("categories.json")
    OUTPUT_FILE = Path("2cat2page_test.json")

    if not CAT_FILE.exists():
        print("[error] categories.json not found — run Cell 9 first")
        return

    cats   = json.loads(CAT_FILE.read_text())["categories"]
    subset = cats[:n_cats]

    print(f"[test] {n_cats} categories × {n_pages} pages max | proxy={'on' if use_proxy else 'off'}\n")

    results = {}
    for i, cat in enumerate(subset, 1):
        key = f"{cat['parent']}/{cat['slug']}"
        print(f"\n{'='*60}")
        print(f"[{i}/{n_cats}] {key}  ({cat['name']})")
        print(f"{'='*60}")

        try:
            await bootstrap_arrow(cat["slug"], use_proxy=use_proxy)
            products = await paginate_with_cookies(cat, max_pages=n_pages)
            results[key] = {"name": cat["name"], "slug": cat["slug"], "count": len(products), "products": products}

        except Exception as e:
            print(f"[error] {key}: {e}")
            results[key] = {"name": cat["name"], "slug": cat["slug"], "count": 0, "products": []}

        finally:
            await close_browser()
            await asyncio.sleep(2)

        # ── write after every category so progress is never lost ─
        OUTPUT_FILE.write_text(json.dumps(results, indent=2))
        print(f"[save] → {OUTPUT_FILE} ({results[key]['count']} products for {key})")

    # ── summary ──────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("[test summary]")
    total = 0
    for key, v in results.items():
        print(f"  {key}: {v['count']} products")
        if v["products"]:
            p = v["products"][0]
            print(f"    sample → {p['partNumber']} | {p['manufacturer']} | ${p['lowestPrice']}")
        total += v["count"]
    print(f"\n  TOTAL: {total} products → {OUTPUT_FILE}")

await run_test(n_cats=2, n_pages=2, use_proxy=True)

[test] 2 categories × 2 pages max | proxy=on


[1/2] amplifiers/active-filters  (Active Filters)
1. Launching Playwright Chromium...
2. Navigating to https://www.arrow.com/en/search-result.html?cat=active-filters (proxy)
3. DOM loaded
4. Waiting 15s for Akamai JS + product API to fire...
  [intercept] ✅ 200 — 25 products | page 1/18 | total 435
5. Page title: Search Result
6. 26 cookies collected
7. API responses captured: 1
8. Products on page 1: 25
   Sample: TLC04CD | Texas Instruments

Browser staying open for pagination cell...
[paginate] 435 products / 2 pages — page 1: 25 products
  [2/2] POST page 2...
  [2] HTTP 200
  [2/2] +25 → 50 total
[closed] browser shut down
[save] → 2cat2page_test.json (50 products for amplifiers/active-filters)

[2/2] amplifiers/amplifiers-amplifier-ic-development-boards-and-kits  (Amplifier IC Development Boards and Kits)
1. Launching Playwright Chromium...
2. Navigating to https://www.arrow.com/en/search-result.html?cat=amplifiers-amplifier-ic-devel

--- 

### Full run, CATS and CAT with all pages

In [57]:
from datetime import datetime, timezone

async def run_single_category(slug: str, use_proxy: bool = True):
    """
    Scrape all pages for a single category by slug.
    Saves to {slug}-1Cat-FullPage.json
    """
    CAT_FILE = Path("categories.json")
    cats     = json.loads(CAT_FILE.read_text())["categories"]

    cat = next((c for c in cats if c["slug"] == slug), None)
    if not cat:
        print(f"[error] slug '{slug}' not found — available sample:")
        for c in cats[:5]:
            print(f"  · {c['slug']} ({c['name']})")
        return

    key      = f"{cat['parent']}/{cat['slug']}"
    out_file = Path(f"{slug}-1Cat-FullPage.json")

    print(f"[run] {key} — {cat['name']} | proxy={'on' if use_proxy else 'off'}")

    try:
        await bootstrap_arrow(cat["slug"], use_proxy=use_proxy)
        products = await paginate_with_cookies(cat, max_pages=None)
    finally:
        await close_browser()

    out = {
        "meta": {
            "category":   key,
            "slug":       cat["slug"],
            "name":       cat["name"],
            "count":      len(products),
            "scraped_at": datetime.now(timezone.utc).isoformat(),
        },
        "products": products,
    }

    out_file.write_text(json.dumps(out, indent=2))
    print(f"[saved] {out_file} — {len(products)} products")

    if products:
        p = products[0]
        print(f"  sample → {p['partNumber']} | {p['manufacturer']} | ${p['lowestPrice']}")

await run_single_category("amplifiers-misc", use_proxy=True)

[run] amplifiers-other/amplifiers-misc — Unspecified Amplifiers | proxy=on
1. Launching Playwright Chromium...
2. Navigating to https://www.arrow.com/en/search-result.html?cat=amplifiers-misc (proxy)
3. DOM loaded
4. Waiting 15s for Akamai JS + product API to fire...
  [intercept] ✅ 200 — 25 products | page 1/9 | total 210
5. Page title: Search Result
6. 40 cookies collected
7. API responses captured: 1
8. Products on page 1: 25
   Sample: AD5750ACPZ | Analog Devices

Browser staying open for pagination cell...
[paginate] 210 products / 9 pages — page 1: 25 products
  [2/9] POST page 2...
  [2] HTTP 200
  [2/9] raw=25 parsed=25 → 50 total
  [3/9] POST page 3...
  [3] HTTP 200
  [3/9] raw=25 parsed=25 → 75 total
  [4/9] POST page 4...
  [4] HTTP 200
  [4/9] raw=25 parsed=25 → 100 total
  [5/9] POST page 5...
  [5] HTTP 200
  [5/9] raw=25 parsed=25 → 125 total
  [6/9] POST page 6...
  [6] HTTP 200
  [6/9] raw=25 parsed=25 → 150 total
  [7/9] POST page 7...
  [7] HTTP 200
  [7/9] raw=25 p